# 00 · Prise en main des données (socle commun)

**Objectif :** comprendre le fichier 2026, définir la population, vérifier ce qui est comparable avec 2025. Les décisions qui en découlent sont dans `DECISIONS.md` et `HARMONISATION.md`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # le notebook vit dans notebooks/ : on importe devia depuis la racine
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160, "display.max_columns", 30)
from devia import data, population
d26 = data.prepare_2026(data.load_2026()); d25 = data.prepare_2025(data.load_2025())
p26, p25 = d26[d26.is_pro_dev], d25[d25.is_pro_dev]          # population par défaut : développeurs professionnels
print(f"2026 : {len(p26):,} développeurs professionnels sur {len(d26):,} répondants | 2025 : {len(p25):,} sur {len(d25):,}")

2026 : 20,220 développeurs professionnels sur 30,903 répondants | 2025 : 34,146 sur 49,191


## 1. Taille et structure
212 colonnes en 2026. Beaucoup de questions ne sont posées qu'à une fraction des répondants (blocs conditionnels) : le taux de réponse varie fortement d'une colonne à l'autre.

In [2]:
raw = data.load_2026()
sch = data.load_schema(2026)
print(sch.shape); print(sch[["column","type","nonmissing"]].head(8).to_string(index=False))
rate = pd.read_csv(population.__file__.replace("population.py","../data/2026/survey_results_public.csv"), nrows=0).columns
print(len(rate), "colonnes dans le CSV")

(212, 10)
        column    type  nonmissing
    ResponseId Derived       30903
           Age      MC       30903
      CodeMain      MC       30794
     Education      MC       30761
    Employment      MC       30903
EmploymentAddl      MC       28099
        ICorPM      MC       24749
   WorkSetting      MC       24602
212 colonnes dans le CSV


## 2. Entonnoir de population
Le commanditaire équipe des salariés : on part de tous les répondants et on garde ceux dont le code est une partie principale du métier, en emploi ou indépendants.

In [3]:
f = pd.DataFrame({"Répondants": [len(d26), (d26.Employment.isin(["Employed","Independent contractor, freelancer, or self-employed"])).sum(), d26.is_pro_dev.sum()]},
  index=["Tous", "En emploi ou indépendants", "… et le code est une partie principale du métier"])
f["Part"] = (f["Répondants"]/len(d26)).round(3); f

,Répondants,Part
Tous,30903,1.000
En emploi ou indépendants,24866,0.805
… et le code est une partie principale du métier,20220,0.654


## 3. Un point d'attention : le taux de réponse au bloc IA
Le bloc sur l'IA n'est pas rempli par tout le monde, et pas dans les mêmes proportions en 2025 et 2026. C'est une limite pour la comparaison.

In [4]:
print("Réponse à la question de capacité (développeurs pros) : 2025 =", round(p25.AIComplex.notna().mean(),3), "| 2026 =", round(p26.AIComplex.notna().mean(),3))
print("Réponse à l'attitude générale : 2025 =", round(p25.AISent.notna().mean(),3), "| 2026 =", round(p26.AISentiment.notna().mean(),3))

Réponse à la question de capacité (développeurs pros) : 2025 = 0.704 | 2026 = 0.504
Réponse à l'attitude générale : 2025 = 0.707 | 2026 = 0.506


**Constat :** le taux de réponse passe d'environ 70 % à environ 50 %. On ne peut pas exclure que les répondants au bloc IA aient changé : c'est écrit dans l'application (onglet *Méthode et limites*).

## 4. Qu'est-ce qui est comparable ?
On lit le schéma 2025 et 2026 pour les questions IA.

In [5]:
s25 = data.load_schema(2025)
print("2025:", [c for c in d25.columns if c.startswith("AI")])
print("2026:", [c for c in d26.columns if c.startswith("AI")][:14], "…")
print(s25[s25.qname.isin(["AISent","AIAcc","AIComplex"])][["qname","question"]].to_string(index=False))

2025: ['AISent', 'AIComplex']
2026: ['AISelect', 'AIFreq_1', 'AIFreq_2', 'AIFreq_3', 'AIFreq_4', 'AIDaily', 'AISentiment', 'AIComplex', 'AITrust', 'AIToolCount', 'AIImpact_1', 'AIImpact_2', 'AIImpact_3', 'AIImpact_4'] …
    qname                                                                                             question
   AISent                 How favorable is your stance on using AI tools as part of your development workflow?
    AIAcc How much do you trust the accuracy of the output from AI tools as part of your development workflow?
AIComplex                  How well do the AI tools you use in your development workflow handle complex tasks?


**Conclusion :** `AIComplex` est identique (même question, mêmes modalités) ; `AISent` / `AISentiment` se ramènent à 3 classes ; `AIAcc` (exactitude) et `AITrust` (confiance dans le flux de travail) sont des questions différentes et ne sont pas comparées.